In [ ]:
 # mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q mlforecast lightgbm xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.7/261.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 22.9 MB/s eta 0:00:00


In [ ]:
import sys,os
sys.path.append('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/src')

In [ ]:
import numpy as np,pandas as pd
#load the model input and save prediction code
import config as cf, data, predictions

In [ ]:
!pip install -q optuna
!pip install -q mlforecast lightgbm

In [ ]:
#chck and build the daily model input file
if os.path.exists(cf.MODEL_INPUT):
    df = data.load()
else:
    df = data.build()

print(len(df), 'rows')

4850000 rows


In [ ]:
#features
import operator
from mlforecast.lag_transforms import (RollingMean,RollingStd,ExpandingMean,ExponentiallyWeightedMean,Combine)

FEATURES=dict(
    lags=[1,7,14,28],
    lag_transforms={
        1:[ExpandingMean(),ExponentiallyWeightedMean(alpha=0.3),Combine(RollingMean(window_size=7),RollingMean(window_size=28),operator.truediv)],
        7: [RollingMean(window_size=7),RollingStd(window_size=7)],
        14:[RollingMean(window_size=14)],
        28:[RollingMean(window_size=28),RollingStd(window_size=28)],
    },
    date_features=['dayofweek','day','is_month_start','is_month_end'],

)

In [ ]:
# Model - direct multi-step forecasting strategy
from mlforecast import MLForecast

def run_ml(model, name, train_df, futr_df, input_version, split):
  train_df = train_df.copy()
  for c in cf.STATIC:
    train_df[c] = train_df[c].astype('category')

  mlmodel=MLForecast(models={'m':model},freq='D',num_threads=os.cpu_count(),**FEATURES)

  # max_horizon=7 fits one model per day ahead
  mlmodel.fit(train_df,static_features=cf.STATIC,dropna=True,max_horizon=cf.HORIZON)

  preds=mlmodel.predict(h=cf.HORIZON,X_df=futr_df)
  if 'unique_id' not in preds.columns:
    preds=preds.reset_index()

  p=preds[['unique_id','ds','m']].rename(columns={'m':'prediction'})
  p[['store_id','product_id']]=p['unique_id'].str.split('_',expand=True).astype(int)
  p['dt']=p['ds']

  predictions.save(p[['store_id','product_id','dt','prediction']],
                   name,input_version,split)

In [ ]:
# the tuning holdout: fit <=06-11, score 06-12..18, on 8,000 series.
FIT_END     = '2024-06-11'
SCORE_START = '2024-06-12'
SCORE_END   = '2024-06-18'
SAMPLE      = 8000

ids=df['unique_id'].drop_duplicates().sample(SAMPLE,random_state=1)
sub=df[df['unique_id'].isin(ids)]
fit_df=sub[sub['ds']<=FIT_END].copy()
fit_df['y']=fit_df['sale_amount_pred']
fit_df=fit_df[['unique_id','ds','y']+cf.EXOG+cf.STATIC]
for c in cf.STATIC:
    fit_df[c]=fit_df[c].astype('category')

scored=sub[(sub['ds']>=SCORE_START)&(sub['ds']<=SCORE_END)]
score_futr=scored[['unique_id','ds']+cf.EXOG]

truth=scored[scored['stock_hour6_22_cnt']==0][['unique_id','ds','sale_amount']]

print('fit rows',len(fit_df),'| scored days',scored['ds'].nunique(),'| truth rows',len(truth))

fit rows 608000 | scored days 7 | truth rows 31452


In [ ]:
# 30 Optuna trials, WAPE on the inner holdout
import optuna
import lightgbm as lgb

def fit_predict(params, train_df, futr_df):
    mlf=MLForecast(models={'m': lgb.LGBMRegressor(**params)}, freq='D',
                     num_threads=os.cpu_count(), **FEATURES)
    mlf.fit(train_df, static_features=cf.STATIC, dropna=True, max_horizon=cf.HORIZON)
    p=mlf.predict(h=cf.HORIZON, X_df=futr_df)
    if 'unique_id' not in p.columns:
        p=p.reset_index()
    return p[['unique_id', 'ds', 'm']].rename(columns={'m': 'prediction'})

def wape(pred):
    m=pred.merge(truth, on=['unique_id', 'ds'], how='inner')
    return (m.prediction - m.sale_amount).abs().sum() / m.sale_amount.abs().sum()

def objective(trial):
    params=dict(
        objective='tweedie',
        tweedie_variance_power=trial.suggest_float('tvp', 1.05, 1.5),
        n_estimators=trial.suggest_int('n_estimators', 200, 800),
        learning_rate=trial.suggest_float('lr', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 31, 255),
        min_child_samples=trial.suggest_int('min_child', 20, 200),
        colsample_bytree=trial.suggest_float('cs', 0.6, 1.0),
        subsample=trial.suggest_float('ss', 0.6, 1.0), subsample_freq=1,
        reg_alpha=trial.suggest_float('l1', 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float('l2', 1e-3, 10, log=True),
        verbosity=-1, random_state=cf.SEED, n_jobs=-1)
    return wape(fit_predict(params, fit_df, score_futr))

# seeded, so the search itself can be reproduced
study=optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('inner-holdout best WAPE:', round(study.best_value, 4))
print('best params:', study.best_params)

[I 2026-08-05 17:50:03,399] A new study created in memory with name: no-name-9ba5ce47-d988-4ba5-83be-7c2710df55de


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-08-05 17:52:45,322] Trial 0 finished with value: 0.33559105080840346 and parameters: {'tvp': 1.218543053481313, 'n_estimators': 771, 'lr': 0.05395030966670229, 'num_leaves': 165, 'min_child': 48, 'cs': 0.662397808134481, 'ss': 0.6232334448672797, 'l1': 2.9154431891537547, 'l2': 0.2537815508265665}. Best is trial 0 with value: 0.33559105080840346.
[I 2026-08-05 17:53:40,584] Trial 1 finished with value: 0.338521922632382 and parameters: {'tvp': 1.3686326600082204, 'n_estimators': 212, 'lr': 0.09330606024425668, 'num_leaves': 218, 'min_child': 58, 'cs': 0.6727299868828402, 'ss': 0.6733618039413735, 'l1': 0.016480446427978974, 'l2': 0.12561043700013558}. Best is trial 0 with value: 0.33559105080840346.
[I 2026-08-05 17:54:45,669] Trial 2 finished with value: 0.3376373452614922 and parameters: {'tvp': 1.2443752583889522, 'n_estimators': 375, 'lr': 0.04091220574443785, 'num_leaves': 62, 'min_child': 72, 'cs': 0.7465447373174767, 'ss': 0.7824279936868144, 'l1': 1.382623217936987, 'l2

In [ ]:
# Cell 7 — the chosen configuration (write these numbers down)
bp = study.best_params
BEST = dict(objective='tweedie', tweedie_variance_power=bp['tvp'],
            n_estimators=bp['n_estimators'], learning_rate=bp['lr'],
            num_leaves=bp['num_leaves'], min_child_samples=bp['min_child'],
            colsample_bytree=bp['cs'], subsample=bp['ss'], subsample_freq=1,
            reg_alpha=bp['l1'], reg_lambda=bp['l2'],
            verbosity=-1, random_state=cf.SEED, n_jobs=-1)
BEST

{'objective': 'tweedie',
 'tweedie_variance_power': 1.4628916449152556,
 'n_estimators': 510,
 'learning_rate': 0.06791196071473647,
 'num_leaves': 227,
 'min_child_samples': 177,
 'colsample_bytree': 0.9568875929449495,
 'subsample': 0.8738943715362388,
 'subsample_freq': 1,
 'reg_alpha': 0.3377170848061959,
 'reg_lambda': 5.053975000590163,
 'verbosity': -1,
 'random_state': 2025,
 'n_jobs': -1}

In [ ]:
#  real sales, validation week
train_df, predict_df = data.splittbl(df, 'raw', 'val')
run_ml(lgb.LGBMRegressor(**BEST), 'LightGBM', train_df, predict_df, 'raw', 'val')

/usr/local/lib/python3.12/dist-packages/mlforecast/lag_transforms.py:1817: RuntimeWarning: invalid value encountered in divide
  return self.operator(self.tfm1.transform(ga), self.tfm2.transform(ga))


[val] 350,000 rows : LightGBM__raw


In [ ]:
# recovered demand, validation week
train_df, predict_df  = data.splittbl(df, 'recovered', 'val')
run_ml(lgb.LGBMRegressor(**BEST), 'LightGBM', train_df, predict_df, 'recovered', 'val')

[val] 350,000 rows : LightGBM__recovered


In [ ]:
# real sales, test week
train_df, predict_df  = data.splittbl(df, 'raw', 'test')
run_ml(lgb.LGBMRegressor(**BEST), 'LightGBM', train_df, predict_df, 'raw', 'test')

/usr/local/lib/python3.12/dist-packages/mlforecast/lag_transforms.py:1817: RuntimeWarning: invalid value encountered in divide
  return self.operator(self.tfm1.transform(ga), self.tfm2.transform(ga))


[test] 350,000 rows : LightGBM__raw


In [ ]:
# recovered demand, test week
train_df, predict_df  = data.splittbl(df, 'recovered', 'test')
run_ml(lgb.LGBMRegressor(**BEST), 'LightGBM', train_df, predict_df, 'recovered', 'test')

[test] 350,000 rows : LightGBM__recovered


In [ ]:
# sanity check
p = pd.read_parquet(f'{cf.PRED_VAL}/LightGBM__raw.parquet')
print('range:', round(p.prediction.min(), 2), '-', round(p.prediction.max(), 2))
print('mean :', round(p.prediction.mean(), 4))

range: 0.0 - 33.18
mean : 1.0516


In [ ]:
import evaluate
long, wide = evaluate.results('val')
print(wide.to_string())

input             raw  recovered    gain
model                                   
PairLGB        0.3107     0.2839  0.0268
LightGBM       0.3180     0.2884  0.0296
TFT            0.3263     0.2895  0.0368
PairTFT        0.3195     0.2900  0.0295
TimesFM        0.3238     0.2974  0.0264
Chronos2       0.3315     0.3050  0.0265
Informer       0.3174     0.3091  0.0083
TiDE           0.3386     0.3133  0.0253
PatchTST       0.3366     0.3147  0.0219
CrostonSBA     0.3572     0.3272  0.0300
LSTM           0.3584     0.3396  0.0188
DLinear        0.3673     0.3397  0.0276
SeasonalNaive  0.3976     0.3739  0.0237


In [ ]:
import evaluate
long, wide = evaluate.results('test')
print(wide.to_string())

input             raw  recovered    gain
model                                   
PairTFT        0.2927     0.2837  0.0090
PairLGB        0.3039     0.2841  0.0198
TFT            0.2947     0.2865  0.0082
TimesFM        0.3045     0.2895  0.0150
TFT_baseline   0.3147     0.2896  0.0251
LightGBM       0.3230     0.2951  0.0279
Chronos2       0.3200     0.2993  0.0207
TiDE           0.3162     0.3093  0.0069
PatchTST       0.3150     0.3116  0.0034
LSTM           0.3139     0.3138  0.0001
CrostonSBA     0.3391     0.3201  0.0190
Informer       0.3062     0.3355 -0.0293
DLinear        0.3534     0.3405  0.0129
SeasonalNaive  0.3891     0.3808  0.0083


In [ ]:
# Cell — WPE by model and pipeline, validation week
import evaluate as ev

long_val, _ = ev.results('val')
wpe_val = long_val.pivot(index='model', columns='input', values='WPE') * 100
wpe_val['shift'] = wpe_val['recovered'] - wpe_val['raw']
print(wpe_val.sort_values('raw').round(2).to_string())

input            raw  recovered  shift
model                                 
LSTM          -18.37       0.04  18.41
TFT           -15.96      -0.90  15.06
CrostonSBA    -14.92      -3.08  11.84
PairTFT       -14.76      -1.21  13.55
DLinear       -13.68      -2.11  11.57
TimesFM       -13.57      -1.53  12.04
TiDE          -13.07      -0.99  12.08
PairLGB       -12.80      -0.82  11.98
PatchTST      -12.31      -0.22  12.09
LightGBM      -12.03      -0.11  11.92
Chronos2      -12.02      -1.00  11.02
SeasonalNaive -10.44       1.78  12.22
Informer       -8.49       6.08  14.57


In [ ]:
# Cell — WPE by model and pipeline, test week
long_test, _ = ev.results('test')
wpe_test = long_test.pivot(index='model', columns='input', values='WPE') * 100
wpe_test['shift'] = wpe_test['recovered'] - wpe_test['raw']
print(wpe_test.sort_values('raw').round(2).to_string())

input            raw  recovered  shift
model                                 
CrostonSBA    -11.16       0.93  12.09
TFT_baseline  -10.53       0.36  10.89
LightGBM      -10.42       0.71  11.13
PairLGB        -9.50       0.41   9.91
Chronos2       -8.84       1.97  10.81
TimesFM        -8.58       0.11   8.69
DLinear        -7.37       4.67  12.04
PairTFT        -7.05       2.02   9.07
TFT            -5.52       3.93   9.45
PatchTST       -5.38       6.84  12.22
TiDE           -5.11       6.93  12.04
LSTM           -4.80       7.53  12.33
SeasonalNaive  -0.77      11.28  12.05
Informer        0.50      16.75  16.25


In [ ]:
# from the csvs
wpe_val  = (pd.read_csv(f'{cf.EVALUATION}/results_val.csv')
              .pivot(index='model', columns='input', values='WPE') * 100)
wpe_test = (pd.read_csv(f'{cf.EVALUATION}/results_test.csv')
              .pivot(index='model', columns='input', values='WPE') * 100)

In [ ]:
# Cell — RMSSE by model and pipeline, validation week
import numpy as np, evaluate as ev, data, config as cf

df = data.load()

def scale_upto(df, end):
    h = df[df['ds'] <= end].sort_values(['unique_id', 'ds'])
    d = h.groupby('unique_id')['sale_amount'].diff(7)
    mse_s = (d ** 2).groupby(h['unique_id']).mean()
    return mse_s[mse_s > 0]

def rmsse_table(split, end):
    mse_s = scale_upto(df, end)
    t, p = ev.truth(split), ev.load_predictions(split)
    m = p.merge(t, on=['store_id', 'product_id', 'dt'], how='inner')
    m['unique_id'] = m['store_id'].astype(str) + '_' + m['product_id'].astype(str)

    rows = []
    for (model, iv), g in m.groupby(['model', 'input']):
        per_sq = ((g['prediction'] - g['sale_amount']) ** 2).groupby(g['unique_id']).mean()
        r = np.sqrt((per_sq / mse_s.reindex(per_sq.index)).dropna()).median()
        rows.append({'model': model, 'input': iv, 'RMSSE': round(r, 3)})
    return pd.DataFrame(rows).pivot(index='model', columns='input', values='RMSSE')

rmsse_val = rmsse_table('val', cf.TUNE_TRAIN_END)     # history to 18 June only
print(rmsse_val.sort_values('recovered').to_string())

input            raw  recovered
model                          
PairLGB        0.648      0.632
PairTFT        0.659      0.639
TFT            0.675      0.639
LightGBM       0.667      0.643
TimesFM        0.662      0.651
LSTM           0.699      0.659
Chronos2       0.688      0.667
TiDE           0.688      0.671
PatchTST       0.683      0.673
CrostonSBA     0.700      0.681
Informer       0.667      0.688
DLinear        0.742      0.726
SeasonalNaive  0.858      0.849


In [ ]:
# RMSSE by model and pipeline, test week
rmsse_test = rmsse_table('test', cf.VAL_END)
print(rmsse_test.sort_values('recovered').to_string())

input            raw  recovered
model                          
PairLGB        0.657      0.654
PairTFT        0.654      0.658
TimesFM        0.663      0.666
TFT            0.664      0.667
TFT_baseline   0.671      0.668
LightGBM       0.681      0.671
Chronos2       0.692      0.682
CrostonSBA     0.702      0.702
PatchTST       0.687      0.705
TiDE           0.694      0.706
LSTM           0.682      0.724
DLinear        0.757      0.769
Informer       0.686      0.781
SeasonalNaive  0.892      0.911


In [ ]:
# RMSSE on the validation week, by model and pipeline
import numpy as np, pandas as pd
import evaluate as ev, data, config as cf

df = data.load()

def scale_upto(df, end):
    """Seasonal-naive squared error per series, using history up to `end` only."""
    h = df[df['ds'] <= end].sort_values(['unique_id', 'ds'])
    d = h.groupby('unique_id')['sale_amount'].diff(7)
    mse_s = (d ** 2).groupby(h['unique_id']).mean()
    return mse_s[mse_s > 0]

def rmsse_table(split, end):
    mse_s = scale_upto(df, end)
    m = ev.load_predictions(split).merge(ev.truth(split),
                                         on=['store_id', 'product_id', 'dt'], how='inner')
    m['unique_id'] = m['store_id'].astype(str) + '_' + m['product_id'].astype(str)

    rows = []
    for (model, iv), g in m.groupby(['model', 'input']):
        per_sq = ((g['prediction'] - g['sale_amount']) ** 2).groupby(g['unique_id']).mean()
        r = np.sqrt((per_sq / mse_s.reindex(per_sq.index)).dropna()).median()
        rows.append({'model': model, 'input': iv, 'RMSSE': round(r, 3)})
    return pd.DataFrame(rows).pivot(index='model', columns='input', values='RMSSE')

rmsse_val = rmsse_table('val', cf.TUNE_TRAIN_END)      # history to 18 June only
rmsse_val['raw_better'] = rmsse_val['raw'] < rmsse_val['recovered']
print(rmsse_val.sort_values('recovered').to_string())
print(f"\nraw better on RMSSE: {rmsse_val['raw_better'].sum()} of {len(rmsse_val)}")

input            raw  recovered  raw_better
model                                      
PairLGB        0.648      0.632       False
PairTFT        0.659      0.639       False
TFT            0.675      0.639       False
LightGBM       0.667      0.643       False
TimesFM        0.662      0.651       False
LSTM           0.699      0.659       False
Chronos2       0.688      0.667       False
TiDE           0.688      0.671       False
PatchTST       0.683      0.673       False
CrostonSBA     0.700      0.681       False
Informer       0.667      0.688        True
DLinear        0.742      0.726       False
SeasonalNaive  0.858      0.849       False

raw better on RMSSE: 1 of 13


In [ ]:
# how recovery shifts each model's bias
wpe_test = (pd.read_csv(f'{cf.EVALUATION}/results_test.csv')
              .pivot(index='model', columns='input', values='WPE') * 100)
wpe_test['shift'] = wpe_test['recovered'] - wpe_test['raw']

print(wpe_test.sort_values('raw').round(2).to_string())
print(f"\nshift  : mean {wpe_test['shift'].mean():.2f}  sd {wpe_test['shift'].std():.2f}")
print(f"Pearson r(raw, recovered) : {wpe_test['raw'].corr(wpe_test['recovered']):.3f}")

input            raw  recovered  shift
model                                 
CrostonSBA    -11.16       0.93  12.09
TFT_baseline  -10.53       0.36  10.89
LightGBM      -10.42       0.71  11.13
Chronos2       -8.84       1.97  10.81
TimesFM        -8.58       0.11   8.69
DLinear        -7.37       4.67  12.04
TFT            -5.52       3.93   9.45
PatchTST       -5.38       6.84  12.22
TiDE           -5.11       6.93  12.04
LSTM           -4.80       7.53  12.33
SeasonalNaive  -0.77      11.28  12.05
Informer        0.50      16.75  16.25

shift  : mean 11.67  sd 1.85
Pearson r(raw, recovered) : 0.956
